# scRareBench | High-Level Harmony | Dataset 3 — Wu breast-cancer atlas

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This notebook runs **Harmony** on scRareBench Dataset 3 through the high-level API:

`load_dataset(3) → MethodSpec(Harmony) → benchmark_method() → HTML / multi-seed bundle`

Pinned scRareBench commit: `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`.

Dataset 3 contract:
- 100,064 cells × 28,468 features
- label: `celltype_subset` (evaluation only; never passed to Harmony)
- batch: `donor_id`
- canonical counts: `layers["counts"]`
- scIB HVG mode: `global`

Method-side preprocessing:
- global Seurat-v3 4,000 HVGs, span 0.3
- normalize_total 10,000 → log1p → scale(max_value=10)
- PCA 50
- Harmony 2.0.0 on `donor_id`

Use a **High-RAM** Colab runtime. Harmony itself does not require a GPU.

## 1. Install pinned scRareBench + Harmony

In [ ]:
import subprocess, sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
SCRAREBENCH_GITHUB = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    SCRAREBENCH_GITHUB,
])

METHOD_DEPENDENCIES = ("harmonypy==2.0.0",)

from scrarebench.runtime import setup_runtime, print_install_report

report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("harmonypy",),
    quiet=False,
)
print_install_report(report)

subprocess.check_call([
    sys.executable,
    "-c",
    "import harmonypy as hm; print('harmonypy', hm.__version__)",
])


## 2. Load Dataset 3

In [ ]:
import scrarebench
from scrarebench import dataset_info, load_dataset

adata = load_dataset(3)
info = dataset_info(adata)

expected = {
    "dataset_key": "wu_breast_cancer_atlas",
    "label_key": "celltype_subset",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "global",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Unexpected Dataset 3 contract for {key}: "
            f"{info.get(key)!r} != {value!r}"
        )

if adata.shape != (100_064, 28_468):
    raise RuntimeError(f"Unexpected Dataset 3 shape: {adata.shape}")
if "counts" not in adata.layers:
    raise RuntimeError("Canonical counts layer is missing.")
if not adata.obs_names.is_unique:
    raise RuntimeError("Cell identifiers are not unique.")

print("scRareBench:", scrarebench.__version__)
print(adata)
print("Donors:", adata.obs["donor_id"].astype(str).nunique())
print(info)


## 3. Harmony configuration

In [ ]:
METHOD_SEEDS = [42, 123, 2026]
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_span": 0.3,
    "normalize_target_sum": 10_000.0,
    "scale_max_value": 10.0,
    "n_pcs": 50,
    "pca_solver": "arpack",

    "theta": 2.0,
    "lamb": None,
    "sigma": 0.1,
    "nclust": None,
    "tau": 0.0,
    "block_size": 0.05,
    "max_iter_harmony": 10,
    "max_iter_kmeans": 4,
    "epsilon_cluster": 1e-3,
    "epsilon_harmony": 1e-2,
    "alpha": 0.2,
    "batch_prop_cutoff": 1e-5,
    "ncores": 0,
    "verbose": True,
}

OUTPUT_DIR = "/content/scrarebench_harmony_dataset3_highlevel"
METHOD_WORK_DIR = "/content/scrarebench_harmony_dataset3_method_runs"


## 4. User-owned Harmony runner

Dataset-specific preparation remains inside scRareBench. This runner only performs
Harmony's method-owned preprocessing/integration. Biological labels are not used.


In [ ]:
from pathlib import Path
import gc, shutil

import anndata as ad
import harmonypy as hm
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

from scrarebench import MethodOutput, dataset_info


def run_harmony(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("dataset_key") != "wu_breast_cancer_atlas":
        raise RuntimeError("This runner is configured for Dataset 3.")
    if method_info.get("scib_hvg_batch_mode") != "global":
        raise RuntimeError("Dataset 3 must use the validated global HVG policy.")
    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(f"Missing canonical count layer: {count_layer!r}")
    if batch_key not in method_adata.obs.columns:
        raise KeyError(f"Missing Harmony batch key: {batch_key!r}")

    original_barcodes = method_adata.obs_names.astype(str).to_numpy(copy=True)

    # Fixed Dataset 3 global Seurat-v3 HVGs.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )
    mask = method_adata.var["highly_variable"].fillna(False).to_numpy(bool)
    expected_hvg = min(int(config["n_hvg"]), method_adata.n_vars)
    if int(mask.sum()) != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} HVGs, selected {int(mask.sum())}."
        )

    counts = method_adata.layers[count_layer][:, mask]
    counts = (
        counts.tocsr().astype(np.float32)
        if sp.issparse(counts)
        else np.asarray(counts, dtype=np.float32)
    )

    work = ad.AnnData(
        X=counts,
        obs=method_adata.obs.copy(),
        var=method_adata.var.loc[method_adata.var_names[mask]].copy(),
    )
    work.var_names_make_unique()

    if not np.array_equal(work.obs_names.astype(str), original_barcodes):
        raise RuntimeError("Cell order changed before Harmony.")

    sc.pp.normalize_total(
        work,
        target_sum=float(config["normalize_target_sum"]),
    )
    sc.pp.log1p(work)
    sc.pp.scale(work, max_value=float(config["scale_max_value"]))

    n_pcs = min(
        int(config["n_pcs"]),
        work.n_vars - 1,
        work.n_obs - 1,
    )
    if n_pcs < 2:
        raise RuntimeError(f"Too few PCA dimensions: {n_pcs}")

    sc.pp.pca(
        work,
        n_comps=n_pcs,
        zero_center=True,
        svd_solver=str(config["pca_solver"]),
        random_state=int(seed),
    )
    pca = np.asarray(work.obsm["X_pca"], dtype=np.float64)

    if pca.shape != (work.n_obs, n_pcs):
        raise RuntimeError(f"Unexpected PCA shape: {pca.shape}")
    if not np.isfinite(pca).all():
        raise ValueError("PCA contains NaN/Inf.")

    np.random.seed(int(seed))
    ho = hm.run_harmony(
        pca,
        work.obs,
        batch_key,
        theta=float(config["theta"]),
        lamb=config["lamb"],
        sigma=float(config["sigma"]),
        nclust=config["nclust"],
        tau=float(config["tau"]),
        block_size=float(config["block_size"]),
        max_iter_harmony=int(config["max_iter_harmony"]),
        max_iter_kmeans=int(config["max_iter_kmeans"]),
        epsilon_cluster=float(config["epsilon_cluster"]),
        epsilon_harmony=float(config["epsilon_harmony"]),
        alpha=float(config["alpha"]),
        batch_prop_cutoff=float(config["batch_prop_cutoff"]),
        verbose=bool(config["verbose"]),
        random_state=int(seed),
        ncores=int(config["ncores"]),
    )

    # harmonypy 2.0 returns corrected PCs as cells × PCs.
    latent = np.asarray(ho.Z_corr, dtype=np.float32)

    if latent.shape != pca.shape:
        raise RuntimeError(
            f"Unexpected Harmony latent shape: {latent.shape}; "
            f"expected {pca.shape}."
        )
    if not np.isfinite(latent).all():
        raise ValueError("Harmony latent contains NaN/Inf.")

    seed_dir = Path(METHOD_WORK_DIR) / f"seed_{int(seed)}"
    if seed_dir.exists():
        shutil.rmtree(seed_dir)
    seed_dir.mkdir(parents=True, exist_ok=True)

    provenance = {}
    objective = np.asarray(
        getattr(ho, "objective_harmony", []),
        dtype=float,
    ).reshape(-1)
    if objective.size:
        objective_path = seed_dir / "harmony_objective_history.csv"
        pd.DataFrame({
            "harmony_iteration": np.arange(objective.size, dtype=int),
            "objective_harmony": objective,
        }).to_csv(objective_path, index=False)
        provenance["harmony_objective_history"] = objective_path

    print(
        f"Seed {seed}: Harmony latent={latent.shape}; "
        f"HVGs={work.n_vars}; "
        f"batches={work.obs[batch_key].astype(str).nunique()}"
    )

    del ho, work, counts, pca
    gc.collect()

    return MethodOutput(
        latent=latent,
        barcodes=original_barcodes,
        representation_key="X_pca_harmony",
        provenance_files=provenance,
    )


## 5. Run the high-level benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="Harmony",
    runner=run_harmony,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "write_pdf_report": False,
        "include_latent_in_bundle": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
    copy_input_per_seed=True,
)


## 6. Inspect results

In [ ]:
from IPython.display import display

display(result.summary().round(5))
print("Method:", result.method)
print("Seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"\nSeed {seed}")
    print("  representation:", run.representation_key)
    print("  report:", run.interactive_report_path)
    print("  bundle:", run.bundle_path)


## 7. Download final archive

In [ ]:
from pathlib import Path

archive = Path(result.archive_path)
if not archive.exists():
    raise FileNotFoundError(archive)

print("Archive:", archive)
print("Size (MB):", round(archive.stat().st_size / 1024 / 1024, 2))

try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass
